# 03 — Utiliser l'historique

**Question :** connaître les semaines passées d'un PDQ prédit-il mieux que le calendrier seul (boosting du notebook 02, MAE 5,17) ?
Et un modèle appris fait-il mieux qu'une règle simple comme « la moyenne des dernières semaines » ?

## Contrat de prédiction (v2)
**Pour chaque PDQ actif, chaque dimanche soir, estimer le nombre d'actes déclarés de la semaine suivante,
à partir du calendrier ET des semaines passées déjà consolidées.**

| Élément | v1 (notebooks 01–02) | v2 (ce notebook) |
|---|---|---|
| Unité, cible, moment | PDQ × semaine, actes déclarés, dimanche soir | identiques |
| Entrées | PDQ, mois, temps | PDQ, mois + **semaines passées consolidées** |

**Latence :** seules les données d'il y a plus de 14 jours sont sûres. Avec un extrait du jour même, la plus récente semaine complète et consolidée serait s−3.
Mais l'extrait disponible le dimanche date de quelques jours (le fichier actuel s'arrête le 4 novembre pour une prédiction le 8) :
la plus récente semaine complète et consolidée est alors **s−4**. On fixe donc un **horizon de 4 semaines** (`horizon` dans `crimes.toml`) :
le modèle apprend avec exactement l'écart qu'il aura en service, comme le bike apprend avec `lag_1` parce qu'il prédit l'heure suivante.
Utiliser une semaine plus récente serait une fuite d'information : le modèle apprendrait avec des chiffres qu'il n'aura pas au moment de prédire.

In [1]:
import json
import math
import hashlib
import tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)

SEED = cfg["experiment"]["seed"]
DATA_PATH = ROOT / cfg["data"]["path"]
IN = ROOT / "artifacts" / "00_start_here"

panel = pd.read_parquet(IN / "panel.parquet")
meta = json.loads((IN / "meta.json").read_text())
meta_02 = json.loads((ROOT / "artifacts" / "02_arbres" / "meta.json").read_text())

empreinte = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:16]
assert empreinte == meta["source_sha16"], "Le CSV a changé depuis le notebook 00 : relance le 00 d'abord."
assert meta_02["source_sha16"] == meta["source_sha16"], "Le notebook 02 a été fait sur d'autres données : relance le 02."

panel["mois"] = panel["SEMAINE"].dt.month
print("Panel :", panel.shape)
print("À battre (notebook 02) :", meta_02["modele_retenu"], "| MAE validation", meta_02["mae_validation"])

Panel : (17835, 5)
À battre (notebook 02) : boosting | MAE validation 5.17


## Choix fixés avant d'entraîner
Le plus petit lag autorisé est **calculé** à partir de la latence de la config, puis vérifié.

In [2]:
H = cfg["historique"]
LAG_MIN = H["horizon"]
assert LAG_MIN >= math.ceil(cfg["panel"]["lag_days"] / 7) + 1, "Horizon plus court que la latence : semaines non consolidées !"

LAGS = H["lags"]
FENETRES = H["fenetres"]

assert min(LAGS) >= LAG_MIN, f"Un lag de moins de {LAG_MIN} semaines utiliserait des semaines non consolidées."

print("Latence :", cfg["panel"]["lag_days"], "jours | horizon :", LAG_MIN, "semaines → plus petit lag autorisé :", LAG_MIN)
print("Lags :", LAGS, "| Moyennes mobiles sur :", FENETRES, "semaines (à partir du lag", LAG_MIN, ")")

Latence : 14 jours | horizon : 4 semaines → plus petit lag autorisé : 4
Lags : [4, 5, 52] | Moyennes mobiles sur : [4, 12] semaines (à partir du lag 4 )


## Construire l'historique de chaque PDQ
`lag_k` = actes du même PDQ il y a k semaines. `moy_w` = moyenne des w semaines consolidées les plus récentes (de s−4 à s−(w+3)).

In [3]:
panel = panel.sort_values(["PDQ", "SEMAINE"]).reset_index(drop=True)
par_pdq = panel.groupby("PDQ")["y"]

for k in LAGS:
    panel[f"lag_{k}"] = par_pdq.shift(k)

for w in FENETRES:
    panel[f"moy_{w}"] = par_pdq.transform(lambda s: s.shift(LAG_MIN).rolling(w).mean())

HISTOIRE = [f"lag_{k}" for k in LAGS] + [f"moy_{w}" for w in FENETRES]
panel.loc[panel["PDQ"] == "38", ["SEMAINE", "PDQ", "y"] + HISTOIRE].iloc[52:57]

,SEMAINE,PDQ,y,lag_4,lag_5,lag_52,moy_4,moy_12
9277,2016-01-04,38,41,65.0,45.0,36.0,49.50,50.750000
9278,2016-01-11,38,42,45.0,65.0,32.0,51.75,51.416667
9279,2016-01-18,38,28,35.0,45.0,40.0,47.50,50.000000
9280,2016-01-25,38,36,25.0,35.0,38.0,42.50,47.666667
9281,2016-02-01,38,29,41.0,25.0,36.0,36.50,45.916667


## Audit d'une ligne
On vérifie, pour le PDQ 38 et la semaine du 2 janvier 2023, que chaque lag vient de la bonne semaine
**et** que cette semaine était déjà consolidée au moment de prédire.

In [4]:
cible = pd.Timestamp("2023-01-02")
ligne = panel[(panel["PDQ"] == "38") & (panel["SEMAINE"] == cible)].iloc[0]
serie = panel[panel["PDQ"] == "38"].set_index("SEMAINE")["y"]

moment = cible - pd.Timedelta(days=1)
consolide = moment - pd.Timedelta(days=cfg["panel"]["lag_days"])
print("Semaine à prédire :", cible.date(), "| prédiction faite le", moment.date(), "| données sûres jusqu'au", consolide.date())

audit = []
for k in LAGS:
    source = cible - pd.Timedelta(weeks=k)
    fin_source = source + pd.Timedelta(days=6)
    assert ligne[f"lag_{k}"] == serie[source]
    assert fin_source <= consolide, "Cette semaine n'était pas encore consolidée !"
    audit.append({"colonne": f"lag_{k}", "semaine_source": source.date(), "fin_source": fin_source.date(),
                  "valeur": ligne[f"lag_{k}"]})
pd.DataFrame(audit)

Semaine à prédire : 2023-01-02 | prédiction faite le 2023-01-01 | données sûres jusqu'au 2022-12-18


,colonne,semaine_source,fin_source,valeur
0,lag_4,2022-12-05,2022-12-11,39.0
1,lag_5,2022-11-28,2022-12-04,49.0
2,lag_52,2022-01-03,2022-01-09,17.0


## Couverture : quelles semaines ont tout leur historique ?
`lag_52` demande un an de passé : les semaines de 2015 n'en ont pas.

In [5]:
complet = panel[HISTOIRE].notna().all(axis=1)
couverture = pd.DataFrame({
    "avant": panel.groupby("ENSEMBLE").size(),
    "avec_historique": panel[complet].groupby("ENSEMBLE").size(),
})
couverture["perdues"] = couverture["avant"] - couverture["avec_historique"]

panel = panel[complet].copy()
print("Première semaine utilisable :", panel["SEMAINE"].min().date())
couverture

Première semaine utilisable : 2016-01-04


,avant,avec_historique,perdues
ENSEMBLE,,,
test,4234,4234,0
train,12093,10585,1508
validation,1508,1508,0


## Outil d'évaluation
`noter` calcule les mesures à partir de prédictions déjà faites (pour les règles simples) ;
`evaluer` entraîne un modèle puis appelle `noter`.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

METRIQUE = "mae"
GROS_PDQ = meta_02["tranche_decision"]

train = panel[panel["ENSEMBLE"] == "train"]
validation = panel[panel["ENSEMBLE"] == "validation"]

modeles = {}
resultats = []

def noter(nom, pred_validation, pred_train):
    erreur = validation["y"] - pred_validation
    gros = validation["PDQ"].isin(GROS_PDQ)
    resultats.append({
        "modele": nom,
        "mae_validation": mean_absolute_error(validation["y"], pred_validation),
        "rmse_validation": root_mean_squared_error(validation["y"], pred_validation),
        "mae_train": mean_absolute_error(train["y"], pred_train),
        "p90_erreur": erreur.abs().quantile(0.9),
        "mae_gros_pdq": erreur[gros].abs().mean(),
        "biais_gros_pdq": erreur[gros].mean(),
    })
    return pd.DataFrame(resultats).set_index("modele").round(3)

def evaluer(nom, estimateur, numeriques=(), categorielles=()):
    colonnes = list(numeriques) + list(categorielles)
    etapes = []
    if numeriques:
        etapes.append(("num", StandardScaler(), list(numeriques)))
    if categorielles:
        etapes.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(categorielles)))
    modele = make_pipeline(ColumnTransformer(etapes), estimateur)
    modele.fit(train[colonnes], train["y"])
    modeles[nom] = modele
    return noter(nom, modele.predict(validation[colonnes]), modele.predict(train[colonnes]))

## Références historiques (sans apprentissage)
Quatre règles simples : « comme il y a 4 semaines », « comme l'an passé », « moyenne des 4 dernières semaines consolidées », « moyenne des 12 ».

In [7]:
for colonne in [f"lag_{LAG_MIN}", "lag_52", "moy_4", "moy_12"]:
    tableau = noter(f"ref_{colonne}", validation[colonne], train[colonne])
tableau

,mae_validation,rmse_validation,mae_train,p90_erreur,mae_gros_pdq,biais_gros_pdq
modele,,,,,,
ref_lag_4,6.586,8.817,5.920,14.00,10.115,0.315
ref_lag_52,7.051,9.422,6.566,15.00,10.892,3.877
ref_moy_4,5.295,7.200,5.038,11.50,8.127,0.552
ref_moy_12,5.040,6.931,4.985,10.75,8.337,0.369


## Ridge avec l'historique
Les 5 colonnes d'historique (lag_4, lag_5, lag_52, moy_4, moy_12) + PDQ + mois.
Le modèle apprend **combien de poids** donner à chaque morceau du passé, au lieu d'une règle fixe.

In [8]:
from sklearn.linear_model import Ridge

evaluer("ridge_hist", Ridge(alpha=10), numeriques=HISTOIRE, categorielles=["PDQ", "mois"])

,mae_validation,rmse_validation,mae_train,p90_erreur,mae_gros_pdq,biais_gros_pdq
modele,,,,,,
ref_lag_4,6.586,8.817,5.920,14.000,10.115,0.315
ref_lag_52,7.051,9.422,6.566,15.000,10.892,3.877
ref_moy_4,5.295,7.200,5.038,11.500,8.127,0.552
ref_moy_12,5.040,6.931,4.985,10.750,8.337,0.369
ridge_hist,4.997,6.835,4.571,10.349,7.589,2.986


## Boosting avec l'historique
Mêmes entrées, mêmes réglages qu'au notebook 02 : seules les entrées ont changé.

In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor

evaluer("boosting_hist", HistGradientBoostingRegressor(max_iter=120, max_leaf_nodes=15, learning_rate=0.08,
                                                       min_samples_leaf=20, early_stopping=False, random_state=SEED),
        numeriques=HISTOIRE, categorielles=["PDQ", "mois"])

,mae_validation,rmse_validation,mae_train,p90_erreur,mae_gros_pdq,biais_gros_pdq
modele,,,,,,
ref_lag_4,6.586,8.817,5.920,14.000,10.115,0.315
ref_lag_52,7.051,9.422,6.566,15.000,10.892,3.877
ref_moy_4,5.295,7.200,5.038,11.500,8.127,0.552
ref_moy_12,5.040,6.931,4.985,10.750,8.337,0.369
ridge_hist,4.997,6.835,4.571,10.349,7.589,2.986
boosting_hist,4.993,6.881,4.278,10.696,7.790,3.540


## Comparaison et choix (sur la validation)
Trois questions : l'historique aide-t-il (vs notebook 02) ? Un modèle appris bat-il la meilleure règle simple ? De combien ?

In [10]:
tableau = pd.DataFrame(resultats).set_index("modele")
references = [nom for nom in tableau.index if nom.startswith("ref_")]
meilleure_ref = tableau.loc[references, "mae_validation"].idxmin()
choix = tableau["mae_validation"].idxmin()

comparaison = pd.DataFrame({
    "mae_validation": {
        "boosting calendrier (notebook 02)": meta_02["mae_validation"],
        f"meilleure règle ({meilleure_ref})": tableau.loc[meilleure_ref, "mae_validation"],
        f"meilleur modèle ({choix})": tableau.loc[choix, "mae_validation"],
    }
})
gain_ref = tableau.loc[meilleure_ref, "mae_validation"] - tableau.loc[choix, "mae_validation"]
print("Modèle retenu :", choix)
print(f"Gain sur la meilleure règle : {gain_ref:.3f} acte ({gain_ref / tableau.loc[meilleure_ref, 'mae_validation']:.1%})")
comparaison.round(3)

Modèle retenu : boosting_hist
Gain sur la meilleure règle : 0.047 acte (0.9%)


,mae_validation
boosting calendrier (notebook 02),5.170
meilleure règle (ref_moy_12),5.040
meilleur modèle (boosting_hist),4.993


**Constat — Historique.** Connaître les semaines passées consolidées aide : la MAE passe de 5,17 (calendrier seul, notebook 02) à 4,99.
Mais l'essentiel du gain vient d'une règle simple : la moyenne des 12 dernières semaines consolidées (5,04), sans aucun apprentissage.
Boosting (4,993) et Ridge (4,997) sont à égalité et ne la battent que de 0,05 acte (0,9 %).
Selon la règle fixée (plus petite MAE en validation), le boosting est retenu ; la moyenne de 12 semaines reste la référence à battre au test.
Sur les 5 plus gros PDQ, les modèles réduisent l'erreur (7,8 contre 8,3) mais les sous-estiment davantage (biais +3,5 contre +0,4).

## L'erreur grandit-elle avec le temps ?

## Ouverture du test (une seule fois)
Deux candidats annoncés : la meilleure règle (moyenne de 12 semaines) et le modèle retenu (boosting avec historique).
Rappel : le boosting calendrier du notebook 02 faisait 5,19 sur ce même test.

In [11]:
test = panel[panel["ENSEMBLE"] == "test"]
print("Test :", test["SEMAINE"].min().date(), "→", test["SEMAINE"].max().date(), "|", len(test), "lignes")

def mesures_test(nom, pred):
    erreur = test["y"] - pred
    gros = test["PDQ"].isin(GROS_PDQ)
    return {
        "modele": nom,
        "mae_test": mean_absolute_error(test["y"], pred),
        "rmse_test": root_mean_squared_error(test["y"], pred),
        "p90_erreur": erreur.abs().quantile(0.9),
        "mae_gros_pdq": erreur[gros].abs().mean(),
        "biais": erreur.mean(),
        "biais_pdq_21": erreur[test["PDQ"] == "21"].mean(),
    }

m = modeles[choix]
lignes_test = [
    mesures_test(meilleure_ref, test[meilleure_ref.removeprefix("ref_")]),
    mesures_test(choix, m.predict(test[list(m.feature_names_in_)])),
]
resultats_test = pd.DataFrame(lignes_test).set_index("modele")
print("Rappel notebook 02 (calendrier seul) : MAE test", meta_02["mae_test"])
resultats_test.round(3)

Test : 2024-01-01 → 2026-10-12 | 4234 lignes
Rappel notebook 02 (calendrier seul) : MAE test 5.191


,mae_test,rmse_test,p90_erreur,mae_gros_pdq,biais,biais_pdq_21
modele,,,,,,
ref_moy_12,4.942,6.796,10.500,8.240,-0.072,1.148
boosting_hist,4.684,6.495,9.848,7.689,0.310,9.359


In [12]:
diag = test.copy()
diag["erreur"] = diag["y"] - m.predict(diag[list(m.feature_names_in_)])
par_annee_test = diag.groupby(diag["SEMAINE"].dt.year)["erreur"].agg(biais="mean", mae=lambda e: e.abs().mean())
par_annee_test.round(2)

,biais,mae
SEMAINE,,
2024,0.25,4.55
2025,0.23,4.75
2026,0.50,4.78


## Rejeu : utiliser le modèle comme en service
Dimanche 11 octobre 2026 au soir, on prédit la semaine suivante du PDQ 21, avec seulement ce qui était connu ce soir-là.
On prédit d'abord, et on ne regarde le réel qu'ensuite.

In [13]:
semaine = test["SEMAINE"].max()
entree = test[(test["PDQ"] == "21") & (test["SEMAINE"] == semaine)]

print("Prédiction faite le", (semaine - pd.Timedelta(days=1)).date(), "pour la semaine du", semaine.date(), "— PDQ 21")
display(entree[["PDQ", "mois"] + HISTOIRE])

prediction = float(m.predict(entree[list(m.feature_names_in_)])[0])
print(f"Prédiction {choix} : {prediction:.1f} actes | règle {meilleure_ref} : {entree[meilleure_ref.removeprefix('ref_')].iloc[0]:.1f} actes")

Prédiction faite le 2026-10-11 pour la semaine du 2026-10-12 — PDQ 21


,PDQ,mois,lag_4,lag_5,lag_52,moy_4,moy_12
4919,21,10,68.0,77.0,62.0,66.0,70.083333


Prédiction boosting_hist : 57.0 actes | règle ref_moy_12 : 70.1 actes


In [14]:
reel = int(entree["y"].iloc[0])
print(f"Réel : {reel} actes | erreur du modèle : {reel - prediction:+.1f}")

Réel : 73 actes | erreur du modèle : +16.0


## Sauvegarde

In [15]:
import joblib
import numpy as np

OUT = ROOT / "artifacts" / "03_historique"
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump(m, OUT / "modele.joblib")
relu = joblib.load(OUT / "modele.joblib")
X_verif = test[list(relu.feature_names_in_)].head(10)
np.testing.assert_allclose(relu.predict(X_verif), m.predict(X_verif))

tableau.round(3).to_csv(OUT / "resultats_validation.csv")
resultats_test.round(3).to_csv(OUT / "resultats_test.csv")
par_annee_test.round(3).to_csv(OUT / "erreur_test_par_annee.csv")

meta_03 = {
    "source_sha16": meta["source_sha16"],
    "contrat": "v2 : PDQ x semaine, dimanche soir, calendrier (PDQ, mois) + semaines passees consolidees",
    "lag_days": cfg["panel"]["lag_days"],
    "lag_min_semaines": LAG_MIN,
    "lags": LAGS,
    "fenetres": FENETRES,
    "colonnes": list(m.feature_names_in_),
    "metrique": METRIQUE,
    "choix_sur": "validation",
    "modele_retenu": choix,
    "reference": meilleure_ref,
    "mae_validation": round(float(tableau.loc[choix, "mae_validation"]), 3),
    "mae_validation_reference": round(float(tableau.loc[meilleure_ref, "mae_validation"]), 3),
    "mae_test": round(float(resultats_test.loc[choix, "mae_test"]), 3),
    "mae_test_reference": round(float(resultats_test.loc[meilleure_ref, "mae_test"]), 3),
    "test_regarde": True,
    "modifie_apres_test": False,
}
(OUT / "meta.json").write_text(json.dumps(meta_03, indent=2, ensure_ascii=False))
sorted(p.name for p in OUT.iterdir())

['erreur_test_par_annee.csv',
 'meta.json',
 'modele.joblib',
 'resultats_test.csv',
 'resultats_validation.csv']

## Résumé

| Approche | MAE validation (2023) | MAE test (2024 → 2026) | Biais PDQ 21 (test) |
|---|---|---|---|
| Boosting calendrier seul (notebook 02) | 5,17 | 5,19 | +13,7 |
| Règle : moyenne des 12 dernières semaines consolidées | 5,04 | 4,94 | **+1,1** |
| **Boosting : historique + PDQ + mois** | **4,99** | **4,68** | +9,4 |

**Ce qu'on retient.**
- Connaître le passé consolidé de chaque PDQ réduit l'erreur de test de 5,19 à **4,68 actes par PDQ-semaine (−10 %)**, et l'erreur grandit peu avec le temps (4,55 → 4,78 sur trois ans).
- La latence et l'âge de l'extrait imposent un horizon de **4 semaines** ; c'est vérifié par un `assert` et par un audit. Avec un horizon de 3 (extrait du jour même), le test aurait été de 4,61 : une semaine de recul ne coûte que 0,08 acte.
- Une règle simple (moyenne de 12 semaines) capte presque tout le gain ; le boosting ne la bat que de 0,05 acte en validation et de 0,26 au test.

**Limites connues.**
- **Niveau du PDQ figé :** le modèle garde un niveau par PDQ appris avant 2023 ; il sous-estime le PDQ 21 (+9,4 au test) après la fermeture du PDQ 22, alors que la règle de 12 semaines suit le nouveau territoire (+1,1).
- **Rejeu :** sur la dernière semaine du PDQ 21, la règle (70,1) est plus proche du réel (73) que le boosting (57,0). Un exemple ne prouve rien, mais il illustre la limite précédente.
- **Délai de 4 semaines :** un changement brusque n'est visible qu'après quatre semaines.
- **Calendrier et historique seulement :** aucun événement (fête, météo, opération policière) n'est connu du modèle.